In [1]:
import numpy as np

import pandas as pd

from astropy.coordinates import SkyCoord

from vasttools.query import Query

In [2]:
import logging
logging.basicConfig(level=201)

In [3]:
psrs = pd.read_csv('paper_df_w_scin_data.csv')

In [4]:
controls = pd.read_csv('final_products/all_final_controls_postprocessed_UPDATED.csv')
raw_psr_meas = pd.read_csv('all_pulsar_measurements.csv')

In [11]:
raw_psr_meas.rename(columns={raw_psr_meas.columns[0]: 'psr_name'}, inplace=True)

In [12]:
corrected_psr_meas = raw_psr_meas.copy()

In [17]:
corrected_flux_list = []
corrected_err_list = []
corrected_flux_series = pd.Series(dtype='Float64')
corrected_err_series = pd.Series(dtype='Float64')
for name in psrs['JNAME'].values:
    print(name)
    control_source = controls[controls['PSR_assoc']==name].iloc[0]
    mycoord = SkyCoord(str(control_source['ra_deg_cont']) + " " + str(control_source['dec_deg_cont']), unit='deg')
    my_query = Query(coords=mycoord, epochs='all-vast', use_tiles=True, corrected_data=False)
    my_query.find_sources()
    
    control_meas = my_query.results[0].measurements
    control_meas['epoch'] = control_meas['epoch'].apply(int)
    psr_meas = raw_psr_meas[raw_psr_meas['psr_name']==name]
    
    control_meas = control_meas[control_meas['detection']]
    psr_meas = psr_meas[psr_meas['detection']]
    
    epochs = np.intersect1d(control_meas['epoch'], psr_meas['epoch'])
    control_mask = (control_meas['epoch'].isin(epochs))
    psr_mask = (psr_meas['epoch'].isin(epochs))
    
    psr_meas = psr_meas[psr_mask]
    control_meas = control_meas[control_mask]
    
    for i in epochs:
        if (psr_meas[psr_meas['epoch']==i].shape[0])>(control_meas[control_meas['epoch']==i].shape[0]):
            size = control_meas[control_meas['epoch']==i].shape[0]
            psr_meas.drop(psr_meas[psr_meas['epoch']==i].index[size:], inplace=True)
        elif (psr_meas[psr_meas['epoch']==i].shape[0])<(control_meas[control_meas['epoch']==i].shape[0]):
            size = psr_meas[psr_meas['epoch']==i].shape[0]
            control_meas.drop(control_meas[control_meas['epoch']==i].index[size:], inplace=True)
    control_meas = pd.DataFrame(data=control_meas.values, columns=control_meas.columns, index=psr_meas.index)
    
    flux_ratios = psr_meas['flux_peak'].div(control_meas['flux_peak'])
    errs = (psr_meas['rms_image'].div(psr_meas['flux_peak']).pow(2) + control_meas['rms_image'].div(control_meas['flux_peak']).pow(2)).pow(0.5).mul(flux_ratios)
    corrected_flux_list.append(flux_ratios)
    corrected_err_list.append(errs) 
    corrected_flux_series = pd.concat([corrected_flux_series, flux_ratios])
    corrected_err_series = pd.concat([corrected_err_series, errs])

J1745-3040
J1804-2717
J1722-3207
J1809-1943
J1720-2933
J1801-2920
J1804-2858
J1759-2205
J1753-1914
J1811-2405
J1740-3015
J1801-2304
J1759-3107
J1802-2124
J1750-3157
J1816-2650
J1743-3150
J1749-3002
J1808-2057
J1807-2715
J1708-3506
J1738-3211
J1739-2903
J1801-3210
J1809-2109
J1759-2922
J1813-2621
J1810-2005
J1727-2951
J1721-3532
J1756-2251


In [21]:
raw_psr_meas['flux_peak'] = corrected_flux_series

raw_psr_meas['rms_image'] = corrected_err_series

In [24]:
raw_psr_meas.drop(columns=raw_psr_meas.columns[1]).to_csv('all_psr_measurements_corrected.csv')